To demonstrate the full machine learning pipeline—**EDA, Feature Engineering, Model Training, and Model Evaluation**—we'll predict house prices based on a dataset containing numerical, categorical, and missing values.

---

**Problem Statement**

Predict `Price` given four input features: `SqFt` (numeric), `Bedrooms` (numeric with missing values), `Location` (categorical), and `Age` (numeric).

---

**1. Exploratory Data Analysis (EDA)**

We load the dataset, check structural integrity, inspect missing values, and analyze summary statistics and feature distributions.

In [ ]:
import numpy as np
import pandas as pd

# 1. Create a raw dataset with missing values and categorical data
raw_data = {
    'SqFt': [1500, 2100, 1200, 2800, 1800, 2500, np.nan, 3000, 1400, 2200],
    'Bedrooms': [3, 4, 2, np.nan, 3, 4, 2, 5, 2, 3],
    'Location': ['Suburbs', 'City', 'Suburbs', 'Downtown', 'Suburbs', 'Downtown', 'Suburbs', 'Downtown', 'Suburbs', 'City'],
    'Age': [10, 5, 20, 2, 15, 8, 12, 1, 25, 6],
    'Price': [250000, 380000, 180000, 520000, 290000, 460000, 210000, 580000, 190000, 400000]
}
df = pd.DataFrame(raw_data)

# 2. Inspect shape and data types
print("--- Data Structure ---")
print(df.info())

# 3. Check for missing values
print("\n--- Missing Values Count ---")
print(df.isnull().sum())

# 4. Summary statistics
print("\n--- Numerical Summary ---")
print(df.describe())

---

**2. Feature Engineering**

We handle missing values (imputation), encode categorical strings into numbers (One-Hot Encoding), and scale numerical values so features share a common range.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Separate features (X) and target variable (y)
X = df.drop(columns=['Price'])
y = df['Price']

# Define feature types
num_features = ['SqFt', 'Bedrooms', 'Age']
cat_features = ['Location']

# 1. Pipeline for numerical features: Impute missing values with median, then scale
# 2. Pipeline for categorical features: One-Hot Encode location values
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first'), cat_features)
    ]
)

# Fit and transform feature matrix
X_processed = preprocessor.fit_transform(X)

# Handle residual NaNs from imputation if using standalone transformers
imputer = SimpleImputer(strategy='median')
X_imputed_num = imputer.fit_transform(X[num_features])
scaler = StandardScaler()
X_scaled_num = scaler.fit_transform(X_imputed_num)

encoder = OneHotEncoder(drop='first', sparse_output=False)
X_encoded_cat = encoder.fit_transform(X[cat_features])

# Combine processed numerical and categorical arrays
X_final = np.hstack((X_scaled_num, X_encoded_cat))
print("Processed Feature Shape:", X_final.shape)


---

**3. Model Training**

We split the processed dataset into training and test sets, select a Linear Regression algorithm, and train the model using `.fit()`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# 1. Split into 80% Training and 20% Testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42
)

# 2. Instantiate the linear regression model
model = LinearRegression()

# 3. Train the model on training data
model.fit(X_train, y_train)

print("Model training complete.")

---

**4. Model Evaluation**

We generate predictions on the unseen test dataset (`X_test`) and evaluate model performance using RMSE and $R^2$ Score.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

# 1. Predict target values on unseen test set
y_pred = model.predict(X_test)

# 2. Calculate evaluation metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("--- Model Performance ---")
print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
print(f"R² Score: {r2:.4f}")

# 3. Compare actual vs predicted values
comparison = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred})
print("\n--- Actual vs Predicted ---")
print(comparison)